In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# CHSH inequality

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Tinatayang paggamit: Dalawang minuto sa isang Heron r3 processor (PAALALA: Ito ay tantya lamang. Maaaring mag-iba ang iyong runtime.)*
## Mga layunin sa pagkatuto
Pagkatapos makumpleto ang tutorial na ito, inaasahang mauunawaan mo ang mga sumusunod:
- Paano gumawa ng parameterized Bell-state CHSH Circuit at sukatin ang apat na expectation values na bumubuo sa mga CHSH witness.
- Paano kalkulahin ang mga expectation values ng maraming observables sa isang parameter sweep sa isang tawag sa [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) primitive.
- Paano i-validate ang isang quantum workflow sa isang maingay na lokal na simulator gamit ang `AerSimulator.from_backend` bago isumite sa hardware.
- Paano palakihin ang isang CHSH experiment sa isang device-wide entanglement benchmark sa pamamagitan ng pagpapatakbo ng maraming independiyenteng Bell pair nang magkasamang-sabay sa IBM Quantum&reg; hardware.

## Mga kinakailangan
Inirerekomenda na pamilyarisado ka sa mga paksang ito:
- [Entanglement in action](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game), isang aralin sa kurso tungkol sa mga Bell state at ang CHSH game.
- [`SparsePauliOp`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) at ang Qiskit primitives [panimula](/guides/primitives).
## Pinagmulan
Sa tutorial na ito, magsasagawa ka ng eksperimento sa isang quantum computer upang ipakita ang paglabag sa CHSH inequality gamit ang Estimator primitive.

Ang CHSH inequality, na pinangalanan mula kina Clauser, Horne, Shimony, at Holt, ay ginagamit upang eksperimentong patunayan ang teorama ni Bell (1969). Inaangkin ng teoremang ito na ang mga lokal na teoryang may nakatagong variable ay hindi makapaliwanag ng ilang kahihinatnan ng entanglement sa quantum mechanics. Ang pagpapakita ng paglabag sa CHSH inequality ay nagpapatunay na ang quantum mechanics ay hindi tugma sa mga lokal na teoryang may nakatagong variable — isang eksperimentong pundamental sa ating pag-unawa sa quantum mechanics.

Ang 2022 Nobel Prize for Physics ay iginawad kina Alain Aspect, John Clauser, at Anton Zeilinger dahil sa kanilang pioneering work sa quantum information science, at lalo na, para sa kanilang mga eksperimento na may entangled photons na nagpapakita ng paglabag sa mga inequality ni Bell.

Para sa eksperimentong ito, gagawa tayo ng entangled pair kung saan susukatin ang bawat qubit sa dalawang magkaibang bases. Tatakan natin ang mga bases para sa unang qubit na $A$ at $a$ at ang mga bases para sa pangalawang qubit na $B$ at $b$. Ito ay nagbibigay-daan sa atin na kalkulahin ang CHSH quantity na $S_1$:

$$
S_1 = A(B-b) + a(B+b).
$$

Ang bawat observable ay $+1$ o $-1$. Malinaw, ang isa sa mga termino na $B\pm b$ ay dapat na $0$, at ang isa pa ay dapat na $\pm 2$. Samakatuwid, $S_1 = \pm 2$. Ang average value ng $S_1$ ay dapat sumunod sa inequality:

$$
|\langle S_1 \rangle|\leq 2.
$$

Ang pag-expand ng $S_1$ sa mga termino ng $A$, $a$, $B$, at $b$ ay nagbibigay ng:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2.
$$

Maaari kang magtalaga ng isa pang CHSH quantity na $S_2$:

$$
S_2 = A(B+b) - a(B-b),
$$

na humahantong sa isa pang inequality:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2.
$$

Kung ang quantum mechanics ay maaaring ilarawan ng mga lokal na teoryang may nakatagong variable, ang mga inequality na ito ay palaging mananatiling totoo. Gaya ng ipinakita sa tutorial na ito, maaari silang labagin sa isang quantum computer, kaya ang quantum mechanics ay hindi tugma sa mga lokal na teoryang may nakatagong variable.

Gagawa tayo ng entangled pair sa pamamagitan ng paghahanda ng Bell state na $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$. Gamit ang Estimator primitive, direkta nating makukuha ang mga expectation values na $\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$, at $\langle ab \rangle$, nang hindi kailangang buuin mula sa mga raw counts. Susukat tayo ng pangalawang qubit sa $Z$ at $X$ bases. Ang unang qubit ay susukatin din sa orthogonal bases, ngunit may rotation angle na $\theta$ na ating i-sweep sa pagitan ng $0$ at $2\pi$. Sinusuri ng Estimator primitive ang parameter sweep na ito sa isang [primitive unified bloc (PUB)](/guides/primitive-input-output).
## Mga kinakailangan
Bago magsimula ng tutorial na ito, tiyaking mayroon kang sumusunod na naka-install:

- Qiskit SDK v2.0 o mas bago, na may [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization) support
- Qiskit Runtime v0.40 o mas bago (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.17 o mas bago (`pip install qiskit-aer`)
## Setup

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Qiskit Aer for local noisy simulation
from qiskit_aer import AerSimulator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

In [ ]:
# Select an IBM Quantum backend.
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=127, operational=True, simulator=False
)
backend.name

'ibm_pittsburgh'

## Small-scale simulator example

Before submitting a hardware job, we validate the entire workflow on a local noisy simulator. We use `AerSimulator.from_backend(backend)` to build a simulator that inherits the noise model and coupling map of the backend you selected, so the simulator response is qualitatively similar to what we expect from hardware.

### Step 1: Map classical inputs to a quantum problem

We write the CHSH circuit with a single parameter $\theta$, which sweeps the measurement basis of the first qubit. The [`Estimator`](/docs/api/qiskit-ibm-runtime/estimator-v2) primitive simplifies the analysis: it returns expectation values of observables directly, and it can evaluate a parameterized circuit at many parameter values in a single call.

In [3]:
theta = Parameter(r"$\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/c3f57d25-0.avif" alt="Output of the previous code cell" />

## Maliit na halimbawa ng simulator
Bago magsumite ng hardware job, ina-validate natin ang buong workflow sa isang lokal na maingay na simulator. Ginagamit natin ang `AerSimulator.from_backend(backend)` upang bumuo ng simulator na nagmamana ng noise model at coupling map ng backend na iyong pinili, kaya ang tugon ng simulator ay kalidad na katulad ng inaasahan natin mula sa hardware.
### Hakbang 1: I-map ang mga classical input sa isang quantum problem
Isinusulat natin ang CHSH Circuit na may isang parameter na $\theta$, na nag-i-sweep ng measurement basis ng unang qubit. Pinasimple ng [`Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) primitive ang pagsusuri: direkta nitong ibinabalik ang mga expectation values ng mga observable, at kaya nitong suriin ang isang parameterized circuit sa maraming parameter values sa isang tawag.

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as a list of lists for the Estimator PUB
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/c3f57d25-0.avif)

Susunod, gagawa tayo ng listahan ng 21 phase values mula $0$ hanggang $2\pi$ kung saan susuriin ang parameterized circuit ($0$, $0.1\pi$, $0.2\pi$, ..., $1.9\pi$, $2\pi$).

In [5]:
# <S_1> = <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <S_2> = <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

Sa wakas, tinutukoy natin ang mga observable. Ang unang qubit ay sinusukat sa mga axes na inikot ng $\theta$; ang pangalawang qubit ay sinusukat sa $Z$ at $X$. Sa mga pagpipiliang ito, ang apat na CHSH correlators ay namamapa sa mga Pauli operator na $ZZ$, $ZX$, $XZ$, at $XX$:

$$
\langle S_1 \rangle = \langle ZZ \rangle - \langle ZX \rangle + \langle XZ \rangle + \langle XX \rangle,
$$

$$
\langle S_2 \rangle = \langle ZZ \rangle + \langle ZX \rangle - \langle XZ \rangle + \langle XX \rangle.
$$

In [6]:
# Build a noisy simulator from the ibm_pittsburgh backend
aer_sim = AerSimulator.from_backend(backend)

pm = generate_preset_pass_manager(target=aer_sim.target, optimization_level=3)
chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/3e139a89-0.avif" alt="Output of the previous code cell" />

### Hakbang 2: I-optimize ang problema para sa quantum hardware execution
Ang mga V2 primitive ay tumatanggap lamang ng mga circuit at observable na sumusunod sa mga instruction at connectivity na sinusuportahan ng target system (instruction set architecture, o ISA, circuits at observables). Binubuo natin ang `AerSimulator` mula sa backend at ini-transpile laban sa target ng simulator upang ang parehong pass manager ay masanay nang buo.

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/3e139a89-0.avif)

Binabago rin natin ang mga observable upang tumugma sa qubit layout ng transpiled circuit gamit ang `SparsePauliOp.apply_layout`.

In [8]:
# Use the AerSimulator-backed Estimator to validate the workflow locally
estimator_sim = Estimator(mode=aer_sim)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA observables
    individual_phases,  # Parameter values
)

sim_result = estimator_sim.run(pubs=[pub]).result()

### Hakbang 3: Isagawa gamit ang mga Qiskit primitive
Pinapatakbo ang parameter sweep gamit ang `EstimatorV2` sa `aer_sim` mode. Ang `run()` method ng Estimator ay tumatanggap ng isang iterable ng mga PUB. Ang bawat PUB ay may format na `(circuit, observables, parameter_values, precision)`. Ipinapasa natin ang parehong observable nang magkasama upang maibahagi nila ang parehong parameter sweep.

In [9]:
chsh1_sim = sim_result[0].data.evs[0]
chsh2_sim = sim_result[0].data.evs[1]


def plot_chsh(phases, chsh1, chsh2, title):
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(
        phases / np.pi, chsh1, "o-", label=r"$\langle S_1 \rangle$", zorder=3
    )
    ax.plot(
        phases / np.pi, chsh2, "o-", label=r"$\langle S_2 \rangle$", zorder=3
    )

    # classical bound +-2
    ax.axhline(y=2, color="0.9", linestyle="--")
    ax.axhline(y=-2, color="0.9", linestyle="--")

    # quantum bound, +-2*sqrt(2)
    ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
    ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
    ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
    ax.fill_between(
        phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7
    )

    ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
    ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel("CHSH witness")
    ax.set_title(title)
    ax.legend()
    plt.show()


plot_chsh(
    phases,
    chsh1_sim,
    chsh2_sim,
    "CHSH witnesses from AerSimulator (ibm_pittsburgh noise model)",
)

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/c8fd5140-0.avif" alt="Output of the previous code cell" />

### Hakbang 4: I-post-process at ibalik ang resulta sa nais na classical format
Ibinabalik ng Estimator ang mga expectation values para sa parehong observable. Ini-plot natin ang mga ito laban sa $\theta$ kasama ang classical bound ($\pm 2$) at ang Tsirelson bound ($\pm 2\sqrt{2}$). Ang mga shaded grey na lugar ay nagmamarka ng agwat sa pagitan ng dalawa. Ang mga puntong nasa loob ng mga band na ito ay lumalabag sa CHSH inequality.

In [10]:
# -------------------------Step 1: Map classical inputs to a quantum problem-------------------------
# A CHSH test is bipartite, so we scale up by running one independent CHSH
# experiment on every disjoint Bell pair the device can host. A greedy
# matching of the coupling map gives a set of edges that share no qubits.
num_qubits = backend.num_qubits
used = set()
pairs = []
for qa, qb in backend.coupling_map.get_edges():
    if qa not in used and qb not in used:
        pairs.append((qa, qb))
        used.update((qa, qb))
num_pairs = len(pairs)
print(
    f"Tiling {backend.name} with {num_pairs} parallel Bell pairs "
    f"({2 * num_pairs} of {num_qubits} qubits)"
)

# One parameterized CHSH sub-circuit per pair, all sharing the angle theta
theta = Parameter(r"$\theta$")
chsh_circuit = QuantumCircuit(num_qubits)
for qa, qb in pairs:
    chsh_circuit.h(qa)
    chsh_circuit.cx(qa, qb)
    chsh_circuit.ry(theta, qa)

# Embed the two CHSH observables onto each pair's qubits (identity elsewhere)
obs1 = SparsePauliOp.from_list([("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)])
obs2 = SparsePauliOp.from_list([("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)])
observables = []
for qa, qb in pairs:
    observables.append([obs1.apply_layout([qa, qb], num_qubits)])
    observables.append([obs2.apply_layout([qa, qb], num_qubits)])

number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
individual_phases = [[ph] for ph in phases]

# -------------------------Step 2: Optimize problem for quantum hardware execution-------------------------
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
chsh_isa_circuit = pm.run(chsh_circuit)
isa_observables = [
    [o[0].apply_layout(chsh_isa_circuit.layout)] for o in observables
]

# -------------------------Step 3: Execute using Qiskit primitives-------------------------
estimator_hw = Estimator(mode=backend)
estimator_hw.options.environment.job_tags = ["TUT_CI"]

pub = (chsh_isa_circuit, isa_observables, individual_phases)
job = estimator_hw.run(pubs=[pub])
print(f"Job ID: {job.job_id()}")
hw_result = job.result()

# -------------------------Step 4: Post-process and return result in desired classical format-------------------------
# evs has shape (2 * num_pairs, number_of_phases); rows alternate S1, S2
evs = np.asarray(hw_result[0].data.evs)
chsh1_all = evs[0::2]
chsh2_all = evs[1::2]

# A pair "violates" CHSH if its strongest witness exceeds the classical bound
peak = np.maximum(
    np.abs(chsh1_all).max(axis=1), np.abs(chsh2_all).max(axis=1)
)
n_violate = int(np.sum(peak > 2))
print(
    f"{n_violate}/{num_pairs} Bell pairs violated the CHSH inequality "
    f"(mean peak witness {peak.mean():.2f}, classical bound 2)"
)

fig, ax = plt.subplots(figsize=(10, 6))

# Faint individual per-pair curves
for row in chsh1_all:
    ax.plot(phases / np.pi, row, color="#1f77b4", alpha=0.2, lw=1)
for row in chsh2_all:
    ax.plot(phases / np.pi, row, color="#ff7f0e", alpha=0.2, lw=1)

# Bold mean curves across all pairs
ax.plot(
    phases / np.pi,
    chsh1_all.mean(axis=0),
    color="#1f77b4",
    lw=2.5,
    label=r"$\langle S_1 \rangle$ (mean)",
)
ax.plot(
    phases / np.pi,
    chsh2_all.mean(axis=0),
    color="#ff7f0e",
    lw=2.5,
    label=r"$\langle S_2 \rangle$ (mean)",
)

# classical bound +-2 and Tsirelson bound +-2*sqrt(2)
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))
ax.set_xlabel(r"$\theta$")
ax.set_ylabel("CHSH witness")
ax.set_title(
    f"CHSH witnesses for {num_pairs} parallel Bell pairs on {backend.name}"
)
ax.legend()
plt.show()

Tiling ibm_pittsburgh with 64 parallel Bell pairs (128 of 156 qubits)
Job ID: d86efd5g7okc73el0rp0
63/64 Bell pairs violated the CHSH inequality (mean peak witness 2.75, classical bound 2)


<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/3376bc73-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/c8fd5140-0.avif)

Ang mga CHSH witness ng simulator ay lumampas na sa classical bound na $\pm 2$ sa ilang mga value ng $\theta$, kahit na may noise model ng backend. Ang mga tuktok ay bahagyang kulang sa Tsirelson bound na $\pm 2\sqrt{2}$ dahil sa simulate na device noise. Sa na-validate na workflow, magpatuloy na tayo sa tunay na hardware.

## Malaking-sukat na halimbawa ng hardware

Ang CHSH test ay mahalagang isang *two-qubit* na eksperimento, kaya hindi ito lumalaki sa pamamagitan ng paggawa ng mas malaking Circuit. Sa halip, lumalaki ito sa pamamagitan ng pagpapatakbo ng **maraming test nang sabay-sabay**. Dito natin inuulit ang backend na may maraming disjoint Bell pair hangga't pinapayagan ng connectivity nito (isang *matching* ng coupling map) at nagpapatakbo ng independiyenteng CHSH sub-circuit sa bawat pair, lahat sa isang job.

Ginagawa nitong isang **device-wide benchmark ng kalidad ng entanglement** ang CHSH: sa halip na isang hand-picked na pair, sinusubukan natin ang entanglement sa malaking bahagi ng chip nang sabay-sabay, sa makatotohanang kondisyon kung saan ang bawat pair ay nakikipaglaban sa crosstalk ng mga kapitbahay at mga parallel-gate error. Ang paglabag sa inequality sa bawat pair *sabay-sabay* ay nagpapatunay na ang tunay na entanglement ay available sa lahat ng dako ng device.